In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

dir = '../../output'
filepath = os.path.join(dir, 'baseflow.dat')

# read baseflow.dat
# each line: radius(n), real(ut(n)), imag(ut(n)), real(oz(n)), imag(oz(n))
if not os.path.exists(filepath):
    print(f"Error: '{filepath}' not found.")
else:
    data = np.loadtxt(filepath)

    r = data[:, 0]
    ut = data[:, 1]
    oz = data[:, 2]
    omega0 = data[:, 3]
    rayleigh_discriminant = data[:, 4]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax1, ax2, ax3, ax4 = axes.flatten()
    rlim = 15

    # Subplot 1: Azimuthal velocity and axial vorticity
    ax1.plot(r, ut, 'b-', label='ut')
    ax1.plot(r, oz, 'r-', label='oz')
    ax1.plot(r, ut/r, 'g-', label='angular velocity ut/r')
    ax1.plot(r, omega0, 'm--', label='angular velocity (from code)')
    ax1.plot(r, rayleigh_discriminant, 'k--', label='Rayleigh Discriminant')
    ax1.set_xlabel('Radius')
    ax1.set_ylabel('Velocity / Vorticity')
    ax1.set_title('Baseflow Profiles')
    ax1.set_xlim(0, rlim)
    ax1.legend()
    ax1.grid(True)

    # Subplot 2: f_eff and sigma_eff
    # handle r=0 case for omega = ut/r
    omega = np.zeros_like(ut)
    if r[0] == 0:
        # at r=0, omega = dut/dr, approximated by forward difference
        if len(r) > 1:
            omega[0] = (ut[1] - ut[0]) / (r[1] - r[0])
            omega[1:] = ut[1:] / r[1:]
        else:
            omega[0] = 0 # Or handle as an error/special case
    else:
        omega = ut / r

    f_eff = 2 * omega
    sigma_eff = -(oz - f_eff)

    ax2.plot(r, f_eff.real, label=r'$f_{eff} = 2\cdot u_\theta/r$')
    ax2.plot(r, sigma_eff.real, label=r'$\sigma_{eff} = -r d_r (u_\theta/r)$')
    ax2.set_xlabel('Radius')
    ax2.set_ylabel('Value')
    ax2.set_title(r'Effective frequencies $f_{eff}$ and $\sigma_{eff}$')
    ax2.set_xlim(0, rlim)
    ax2.legend()
    ax2.grid(True)

    ax3.plot(r, rayleigh_discriminant, 'k-', label='Rayleigh Discriminant')
    ax3.set_xlabel('Radius')
    ax3.set_ylabel('Value')
    ax3.set_title('Rayleigh Discriminant')
    ax3.set_xlim(0, rlim)
    ax3.legend()
    ax3.grid(True)

    ax4.plot(r, omega0, 'm-', label='Angular Velocity ut/r')
    ax4.set_xlabel('Radius')
    ax4.set_ylabel('Value')
    ax4.set_title('Angular Velocity')
    ax4.set_xlim(0, rlim)
    ax4.legend()
    ax4.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
from scipy.linalg import expm

S = np.array([[1j, -1j], [1, 1]])
S_INV = np.array([[-1j/2, 1/2], [1j/2, 1/2]])
print("Transformation matrix S:")
print(S)
print("Inverse transformation matrix S_INV:")
print(S_INV)

# calculate ETD's linear and nonlinear integration operators
# using oz[r[0]] as an example (vortex center)
dt = 0.1
J = np.array([[-oz[0]*1j, 0], [0, oz[0]*1j]])

L = S @ (J @ S_INV)
print("Linear L:")
print(L)

J_EXP = S @ (expm(J*dt) @ S_INV)
print("Exponential J_EXP:")
print(J_EXP)

J_NL = S @ (np.linalg.inv(J) @ ((expm(J*dt)-np.eye(2)) @ S_INV))
print("Non-linear J_NL:")
print(J_NL)

In [ ]:
m = 0
nn = 0
dt = 0.1
Omega = 0.0
N = 0.1

LH = np.array([[-1j*m*omega0[nn], 2*Omega + 2*omega0[nn]], [-2*Omega - oz[nn], -1j*m*omega0[nn]]])
print("Horizontal linear matrix LH:")
print(LH)
LH_EXP = expm(LH*dt)
print("EXP(LH*dt):")
print(LH_EXP)
print("LH^-1@(EXP(LH*dt)-I):")
print(np.linalg.inv(LH) @ (LH_EXP - np.eye(2)))

print("\n")

LV = np.array([[-1j*m*omega0[nn], -1], [N**2, -1j*m*omega0[nn]]])
print("Vertical linear matrix LV:")
print(LV)
LV_EXP = expm(LV*dt)
print("EXP(LV*dt):")
print(LV_EXP)
print("LV^-1@(EXP(LV*dt)-I):")
print(np.linalg.inv(LV) @ (LV_EXP - np.eye(2)))